# Data Cleaning: Amazon Sales Dataset
**Oasis Infobyte | Data Analytics | Level 1 | Task 3**

**Objective:** Convert the raw, messy Amazon India product dataset into a clean, analysis-ready file and document every decision.

## 1. Load the dataset
The raw file is loaded into `raw`, and all work is done on a copy (`df`) so the original data stays untouched.

In [1]:
import numpy as np
import pandas as pd
pd.options.display.float_format = "{:,.2f}".format

raw = pd.read_csv("amazon.csv")
df = raw.copy()
print(df.shape)
df.head(3)

(1465, 16)


,product_id,product_name,category,discounted_price,actual_price,discount_percentage,rating,rating_count,about_product,user_id,user_name,review_id,review_title,review_content,img_link,product_link
0,B07JW9H4J1,Wayona Nylon Braided USB to Lightning Fast Cha...,Computers&Accessories|Accessories&Peripherals|...,₹399,"₹1,099",64%,4.2,"24,269",High Compatibility : Compatible With iPhone 12...,"AG3D6O4STAQKAY2UVGEUV46KN35Q,AHMY5CWJMMK5BJRBB...","Manav,Adarsh gupta,Sundeep,S.Sayeed Ahmed,jasp...","R3HXWT0LRP0NMF,R2AJM3LFTLZHFO,R6AQJGUP6P86,R1K...","Satisfied,Charging is really fast,Value for mo...",Looks durable Charging is fine tooNo complains...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Wayona-Braided-WN3LG1-Sy...
1,B098NS6PVG,Ambrane Unbreakable 60W / 3A Fast Charging 1.5...,Computers&Accessories|Accessories&Peripherals|...,₹199,₹349,43%,4.0,"43,994","Compatible with all Type C enabled devices, be...","AECPFYFQVRUWC3KGNLJIOREFP5LQ,AGYYVPDD7YG7FYNBX...","ArdKn,Nirbhay kumar,Sagar Viswanathan,Asp,Plac...","RGIQEG07R9HS2,R1SMWZQ86XIN8U,R2J3Y1WL29GWDE,RY...","A Good Braided Cable for Your Type C Device,Go...",I ordered this cable to connect my phone to An...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Ambrane-Unbreakable-Char...
2,B096MSW6CT,Sounce Fast Phone Charging Cable & Data Sync U...,Computers&Accessories|Accessories&Peripherals|...,₹199,"₹1,899",90%,3.9,"7,928",【 Fast Charger& Data Sync】-With built-in safet...,"AGU3BBQ2V2DDAMOAKGFAWDDQ6QHA,AESFLDV2PT363T2AQ...","Kunal,Himanshu,viswanath,sai niharka,saqib mal...","R3J3EQQ9TZI5ZJ,R3E7WBGK7ID0KV,RWU79XKQ6I1QF,R2...","Good speed for earlier versions,Good Product,W...","Not quite durable and sturdy,https://m.media-a...",https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Sounce-iPhone-Charging-C...


## 2. Data quality report (before cleaning)
Checking data types, null values, unique counts and duplicates to find problems before fixing anything.

In [2]:
report = pd.DataFrame({"dtype": df.dtypes.astype(str),
                       "nulls": df.isnull().sum(),
                       "unique": df.nunique()})
display(report)
print("Duplicate rows:", df.duplicated().sum())
print("Repeated product_id:", df["product_id"].duplicated().sum())

,dtype,nulls,unique
product_id,str,0,1351
product_name,str,0,1337
category,str,0,211
discounted_price,str,0,550
actual_price,str,0,449
discount_percentage,str,0,92
rating,str,0,28
rating_count,str,2,1143
about_product,str,0,1293
user_id,str,0,1194


Duplicate rows: 0
Repeated product_id: 114


**Issues found:** Price, discount, rating and rating_count are stored as text (₹, commas, %). `rating_count` has 2 missing values. 114 rows repeat a `product_id`. `category` is a pipe-separated hierarchy.

## 3. Save the "before" snapshot
Recording key metrics now so a before vs. after comparison can be made at the end.

In [4]:
num_cols = ["discounted_price", "actual_price",
            "discount_percentage", "rating", "rating_count"]
before = {
    "rows": len(raw),
    "total_nulls": int(raw.isnull().sum().sum()),
    "duplicate_rows": int(raw.duplicated().sum()),
    "duplicate_product_ids": int(raw["product_id"].duplicated().sum()),
    "numeric_cols_correct_dtype": int(sum(
        pd.api.types.is_numeric_dtype(raw[c]) for c in num_cols)),
}
before

{'rows': 1465,
 'total_nulls': 2,
 'duplicate_rows': 0,
 'duplicate_product_ids': 114,
 'numeric_cols_correct_dtype': 0}

## 4. Fix data types (text to numbers)
Removing ₹, commas and % signs, then converting to numbers. `errors="coerce"` turns unparseable values into NaN instead of crashing, so they show up as nulls and get handled in the next step.

In [5]:
def to_number(s):
    return pd.to_numeric(
        s.astype(str).str.replace(r"[₹,%]", "", regex=True).str.strip(),
        errors="coerce")

for col in num_cols:
    df[col] = to_number(df[col])

print(df[num_cols].dtypes)
print(df[num_cols].isnull().sum())

discounted_price       float64
actual_price           float64
discount_percentage      int64
rating                 float64
rating_count           float64
dtype: object
discounted_price       0
actual_price           0
discount_percentage    0
rating                 1
rating_count           2
dtype: int64


## 5. Handle missing values
**Strategy:** median imputation for `rating_count` and `rating`. `rating_count` is highly right-skewed (a few products have very large counts), so the mean would be misleading. Only 3 cells are affected, so imputing keeps the products in the dataset.

In [6]:
df["rating_count"] = df["rating_count"].fillna(df["rating_count"].median())
df["rating"] = df["rating"].fillna(df["rating"].median())
print("Nulls left:", df.isnull().sum().sum())

Nulls left: 0


## 6. Remove duplicates
There are no fully duplicated rows, but the same `product_id` appears multiple times. Product-level analysis needs one row per product, so the first occurrence is kept. 114 rows are removed.

In [7]:
n_before = len(df)
df = df.drop_duplicates(subset="product_id", keep="first").reset_index(drop=True)
print(f"Removed {n_before - len(df)} duplicates -> {len(df)} unique products")

Removed 114 duplicates -> 1351 unique products


In [8]:
for col in ["product_name", "category"]:
    df[col] = df[col].str.strip()

parts = df["category"].str.split("|", expand=True)
df["main_category"] = parts[0].str.replace("&", " & ")
df["sub_category"] = parts[1].str.replace("&", " & ")
df["leaf_category"] = df["category"].str.split("|").str[-1].str.replace("&", " & ")
print(df["main_category"].value_counts())

main_category
Electronics                490
Home & Kitchen             448
Computers & Accessories    375
OfficeProducts              31
MusicalInstruments           2
HomeImprovement              2
Toys & Games                 1
Car & Motorbike              1
Health & PersonalCare        1
Name: count, dtype: int64


## 7. Validate values and add features
Checks: ratings must be 1 to 5, prices must be positive, and the stored discount % must match the one calculated from the prices. Also adding `discount_amount` as a useful new column.

In [9]:
print("Ratings outside 1-5:", ((df["rating"] < 1) | (df["rating"] > 5)).sum())
print("Non-positive prices:", (df["discounted_price"] <= 0).sum())

calc = ((df["actual_price"] - df["discounted_price"])
        / df["actual_price"] * 100).round()
print("Discount mismatches:", ((calc - df["discount_percentage"]).abs() > 1).sum())

df["discount_amount"] = df["actual_price"] - df["discounted_price"]
df["rating_count"] = df["rating_count"].astype(int)

Ratings outside 1-5: 0
Non-positive prices: 0
Discount mismatches: 0


**Result:** All values are valid. No ratings are out of range, no prices are non-positive, and the discount percentages match the prices.

## 8. Outlier detection (IQR method)
Values beyond 1.5 × IQR from the quartiles are flagged as outliers.

In [10]:
rows = []
for col in ["discounted_price", "actual_price", "rating_count", "rating"]:
    q1, q3 = df[col].quantile([.25, .75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n = ((df[col] < lo) | (df[col] > hi)).sum()
    rows.append({"column": col, "lower": lo, "upper": hi,
                 "outliers": n, "outlier_%": n / len(df) * 100})
pd.DataFrame(rows)

,column,lower,upper,outliers,outlier_%
0,discounted_price,"-2,388.50","4,911.50",209,15.47
1,actual_price,"-4,615.00","10,089.00",185,13.69
2,rating_count,"-21,225.00","38,327.00",130,9.62
3,rating,3.30,4.90,19,1.41


**Decision: all outliers are retained.**
- `discounted_price` and `actual_price`: high values are genuine premium products (TVs, laptops), not errors.
- `rating_count`: popular products legitimately have very large counts. A log transform is better than removal for modelling.
- `rating`: low ratings reflect real customer opinion.

## 9. Before vs. after summary

In [11]:
after = {
    "rows": len(df),
    "total_nulls": int(df[num_cols].isnull().sum().sum()),
    "duplicate_rows": int(df.duplicated().sum()),
    "duplicate_product_ids": int(df["product_id"].duplicated().sum()),
    "numeric_cols_correct_dtype": int(sum(
        pd.api.types.is_numeric_dtype(df[c]) for c in num_cols)),
}
pd.DataFrame({"Before": before, "After": after})

,Before,After
rows,1465,1351
total_nulls,2,0
duplicate_rows,0,0
duplicate_product_ids,114,0
numeric_cols_correct_dtype,0,5


## 10. Save the cleaned dataset

In [12]:
df.to_csv("amazon_cleaned.csv", index=False)
print(df.shape)

(1351, 20)


## Conclusion
The raw dataset had text-formatted numbers, missing `rating_count` values and repeated product IDs. After cleaning, all numeric columns have correct data types, nulls are handled, there is one row per product (1,465 to 1,351 rows), categories are split into usable levels, and discounts were validated against prices. The cleaned file `amazon_cleaned.csv` is ready for analysis such as category-wise pricing and rating trends.